# Phase 2 Dataset EDA And Temporal Split Summary

This notebook presents exploratory analysis and the leakage-safe chronological train/validation/test split. It does not train models or extract ML features.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load import DEFAULT_RAW_DATASET_PATH, read_dataset_columns
from src.data.split import build_temporal_split, load_split_config, save_temporal_split

split_metadata_path = PROJECT_ROOT / "results" / "baseline" / "temporal_split_metadata.json"
split_membership_path = PROJECT_ROOT / "results" / "baseline" / "temporal_split_membership.csv"

if not split_metadata_path.exists() or not split_membership_path.exists():
    config = load_split_config(PROJECT_ROOT / "configs" / "data_split.yaml")
    membership, metadata = build_temporal_split(config)
    save_temporal_split(membership, metadata, config.output_dir)

split_metadata = json.loads(split_metadata_path.read_text(encoding="utf-8"))
split_membership = pd.read_csv(split_membership_path)
metadata = read_dataset_columns(
    ["address", "createdBlockNumber", "creator", "label", "timestamp"],
    path=DEFAULT_RAW_DATASET_PATH,
)
metadata["createdBlockNumber"] = pd.to_numeric(metadata["createdBlockNumber"])
metadata["timestamp"] = pd.to_datetime(metadata["timestamp"])
metadata.head()

## Class Distribution

In [ ]:
class_counts = metadata["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count")
class_counts["share"] = class_counts["count"] / class_counts["count"].sum()
class_counts

In [ ]:
ax = class_counts.plot.bar(x="label", y="count", legend=False, color=["#4C78A8", "#F58518"])
ax.set_title("Class distribution")
ax.set_xlabel("Label")
ax.set_ylabel("Contracts")
plt.tight_layout()

## Block Distribution

In [ ]:
ax = metadata["createdBlockNumber"].plot.hist(bins=30, color="#54A24B")
ax.set_title("Contracts by createdBlockNumber")
ax.set_xlabel("createdBlockNumber")
plt.tight_layout()

In [ ]:
metadata.groupby("label")["createdBlockNumber"].agg(["count", "min", "median", "max"]).reset_index()

## Creator Distribution

In [ ]:
creator_counts = metadata["creator"].value_counts()
creator_summary = pd.Series(
    {
        "unique_creators": metadata["creator"].nunique(),
        "creators_with_multiple_contracts": int((creator_counts > 1).sum()),
        "max_contracts_by_one_creator": int(creator_counts.max()),
    },
    name="value",
).rename_axis("measure").reset_index()
creator_summary

In [ ]:
creator_counts.head(15).rename_axis("creator").reset_index(name="contract_count")

## Leakage-Safe Temporal Split Summary

In [ ]:
split_counts = pd.DataFrame.from_dict(split_metadata["split_counts"], orient="index")
split_counts.index.name = "split"
split_counts.reset_index()

In [ ]:
split_label_counts = split_membership.groupby(["split", "label"]).size().unstack(fill_value=0)
ax = split_label_counts.loc[["train", "validation", "test"]].plot.bar(stacked=True, color=["#4C78A8", "#F58518"])
ax.set_title("Class counts by chronological split")
ax.set_xlabel("Split")
ax.set_ylabel("Contracts")
plt.tight_layout()

In [ ]:
split_metadata["temporal_boundary_assertions"]

## Cross-Split Duplicate Signals

In [ ]:
pd.DataFrame.from_dict(split_metadata["cross_split_duplicates"], orient="index").reset_index(names="column")